# Statistical overview for all dataset analyzed in this study

In [ ]:
import pandas as pd
from tabulate import tabulate
from sklearn.model_selection import train_test_split

rows = []

OOD_DATASETS = {
    'Ruby':       '/d/ruby.csv',
    'Go':         '/d/goo.csv',
    'JavaScript': '/d/js.csv',
    'PHP':        '/d/php.csv',
    'Kotlin':     '/d/kot.csv',
    'Fortran':    '/d/for.csv',
    'Java':       '/d/java.csv',
}

def append_split_stats(rows, task, dataset, split_name, df):
    pos = int((df['label'] == 1).sum())
    neg = int((df['label'] == 0).sum())
    rows.append({'Task': task, 'Dataset': dataset, 'Split': split_name,
                 'Positive': pos, 'Negative': neg, 'Total': pos + neg})

def stratified_splits(df):
    train_val, test = train_test_split(df, test_size=0.10, stratify=df['label'], random_state=42)
    train, val = train_test_split(train_val, test_size=2/9, stratify=train_val['label'], random_state=42)
    return train, val, test

for lang, path in OOD_DATASETS.items():
    df = pd.read_csv(path)
    append_split_stats(rows, 'Vuln. Detection (OOD)', f'OOD-{lang}', 'Inference', df)

for task, dataset, path in [
    ('Vuln. Detection (Original)',  'Codex',           'traincodex.csv'),
    ('Vuln. Detection (Processed)', 'Codex-Processed', 'ode vul/trqw.csv'),
]:
    df = pd.read_csv(path)
    train, val, test = stratified_splits(df)
    append_split_stats(rows, task, dataset, 'Train (70%)', train)
    append_split_stats(rows, task, dataset, 'Val   (20%)', val)
    append_split_stats(rows, task, dataset, 'Test  (10%)', test)

df_clone = pd.read_csv('/clone/train.csv')
train_c, val_c, test_c = stratified_splits(df_clone)
append_split_stats(rows, 'Clone Detection', 'BigCloneBench', 'Train (70%)', train_c)
append_split_stats(rows, 'Clone Detection', 'BigCloneBench', 'Val   (20%)', val_c)
append_split_stats(rows, 'Clone Detection', 'BigCloneBench', 'Test  (10%)', test_c)

table_df = pd.DataFrame(rows, columns=['Task', 'Dataset', 'Split', 'Positive', 'Negative', 'Total'])
print(tabulate(table_df, headers='keys', tablefmt='fancy_grid', showindex=False))


# CodeXGLUE dataset statistics with simple preprocessing view

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from transformers import AutoTokenizer
import re
import warnings
warnings.filterwarnings('ignore')

model_name = "microsoft/unixcoder-base"
train_csv = 'l/traincodex.csv'
test_csv = '/testcodex.csv'

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

def clean_code(code):
    code = str(code)
    
    code = re.sub(r'/\*[\s\S]*?\*/', '', code)
    
    code = re.sub(r'//.*?$', '', code, flags=re.MULTILINE)
    
    code = re.sub(r'#.*?$', '', code, flags=re.MULTILINE)
    
    code = re.sub(r'"""[\s\S]*?"""', '', code)
    code = re.sub(r"'''[\s\S]*?'''", '', code)
    
    lines = code.split('\n')
    non_empty_lines = [line.rstrip() for line in lines if line.strip()]
    code = '\n'.join(non_empty_lines)
    
    code = re.sub(r'\n{3,}', '\n\n', code)
    
    code = re.sub(r'[ \t]+', ' ', code)
    
    code = code.strip()
    
    return code

def load_and_inspect_dataset(csv_path, dataset_name):
    print(f"\n{'='*80}")
    print(f"DATASET INSPECTION: {dataset_name}")
    print(f"{'='*80}")
    
    df = pd.read_csv(csv_path)
    
    print(f"\n1. BASIC INFORMATION")
    print(f"   Rows: {len(df):,}")
    print(f"   Columns: {len(df.columns)}")
    print(f"   Column names: {df.columns.tolist()}")
    print(f"   Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    print(f"\n2. DATA TYPES")
    print(df.dtypes)
    
    print(f"\n3. MISSING VALUES")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100)
    missing_df = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
    if missing.sum() > 0:
        print(missing_df[missing_df['Missing'] > 0])
    else:
        print("   No missing values detected")
    
    print(f"\n4. DUPLICATE ROWS")
    duplicates = df.duplicated().sum()
    print(f"   Number of duplicate rows: {duplicates} ({duplicates/len(df)*100:.2f}%)")
    
    if 'label' in df.columns or 'target' in df.columns:
        label_col = 'label' if 'label' in df.columns else 'target'
        print(f"\n5. CLASS DISTRIBUTION")
        print(df[label_col].value_counts())
        print(f"\n   Class Balance:")
        print(df[label_col].value_counts(normalize=True) * 100)
    
    print(f"\n6. SAMPLE DATA (First 2 rows)")
    for idx, row in df.head(2).iterrows():
        print(f"\n   Row {idx}:")
        for col in df.columns:
            value = str(row[col])[:80]
            print(f"     {col}: {value}...")
    
    return df

def analyze_token_lengths(df, dataset_name, text_column, apply_cleaning=True):
    print(f"\n{'='*80}")
    print(f"TOKEN LENGTH ANALYSIS: {dataset_name}")
    if apply_cleaning:
        print(f"WITH CODE PREPROCESSING (removing comments & whitespace)")
    else:
        print(f"WITHOUT CODE PREPROCESSING (raw code)")
    print(f"{'='*80}")
    
    texts = df[text_column].fillna('').astype(str)
    
    if apply_cleaning:
        print(f"\nCleaning and tokenizing {len(texts)} samples...")
        cleaned_texts = []
        for i, text in enumerate(texts):
            cleaned = clean_code(text)
            cleaned_texts.append(cleaned)
            if (i + 1) % 1000 == 0:
                print(f"  Cleaned {i+1}/{len(texts)} samples...")
        texts = cleaned_texts
    else:
        print(f"\nTokenizing {len(texts)} samples (no cleaning)...")
    
    token_lengths = []
    original_lengths = []
    cleaned_lengths = []
    
    for i, text in enumerate(texts):
        tokens = tokenizer.encode(text, add_special_tokens=True)
        token_lengths.append(len(tokens))
        
        if apply_cleaning:
            cleaned_lengths.append(len(text))
            original_lengths.append(len(df[text_column].iloc[i]))
        
        if (i + 1) % 1000 == 0:
            print(f"  Processed {i+1}/{len(texts)} samples...")
    
    token_lengths = np.array(token_lengths)
    
    if apply_cleaning:
        reduction_pct = (1 - np.mean(cleaned_lengths) / np.mean(original_lengths)) * 100
        print(f"\n📊 CLEANING IMPACT:")
        print(f"   Average original code length: {np.mean(original_lengths):.0f} chars")
        print(f"   Average cleaned code length: {np.mean(cleaned_lengths):.0f} chars")
        print(f"   Average reduction: {reduction_pct:.2f}%")
    
    print(f"\n1. BASIC STATISTICS")
    print(f"   Min tokens: {token_lengths.min()}")
    print(f"   Max tokens: {token_lengths.max()}")
    print(f"   Mean tokens: {token_lengths.mean():.2f}")
    print(f"   Median tokens: {np.median(token_lengths):.2f}")
    print(f"   Std tokens: {token_lengths.std():.2f}")
    print(f"   Total tokens: {token_lengths.sum():,}")
    
    print(f"\n2. PERCENTILES")
    percentiles = [25, 50, 75, 90, 95, 99]
    for p in percentiles:
        print(f"   {p}th percentile: {np.percentile(token_lengths, p):.0f} tokens")
    
    print(f"\n3. TOKEN RANGE DISTRIBUTION")
    ranges = [(0, 128), (128, 256), (256, 512), (512, 1024), (1024, 2048), (2048, float('inf'))]
    for start, end in ranges:
        count = np.sum((token_lengths >= start) & (token_lengths < end))
        percentage = (count / len(token_lengths)) * 100
        range_str = f"{start}-{end}" if end != float('inf') else f"{start}+"
        print(f"   {range_str:15s}: {count:6d} samples ({percentage:5.2f}%)")
    
    print(f"\n4. TRUNCATION ANALYSIS")
    max_lengths = [128, 256, 512, 1024]
    for max_len in max_lengths:
        truncated = np.sum(token_lengths > max_len)
        pct = (truncated / len(token_lengths)) * 100
        print(f"   Samples > {max_len} tokens: {truncated:6d} ({pct:5.2f}%)")
    
    return token_lengths

def create_comparison_visualizations(raw_train_tokens, raw_test_tokens, 
                                    clean_train_tokens, clean_test_tokens,
                                    train_df, test_df):
    print(f"\n{'='*80}")
    print(f"CREATING COMPARISON VISUALIZATIONS")
    print(f"{'='*80}")
    
    sns.set_style("whitegrid")
    sns.set_palette("husl")
    
    fig = plt.figure(figsize=(24, 20))
    
    plt.subplot(4, 3, 1)
    plt.hist(raw_train_tokens, bins=50, alpha=0.7, label='Raw', edgecolor='black', color='coral')
    plt.hist(clean_train_tokens, bins=50, alpha=0.7, label='Cleaned', edgecolor='black', color='skyblue')
    plt.xlabel('Token Length', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.title('Train: Raw vs Cleaned Token Distribution', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(4, 3, 2)
    plt.hist(raw_test_tokens, bins=50, alpha=0.7, label='Raw', edgecolor='black', color='coral')
    plt.hist(clean_test_tokens, bins=50, alpha=0.7, label='Cleaned', edgecolor='black', color='skyblue')
    plt.xlabel('Token Length', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.title('Test: Raw vs Cleaned Token Distribution', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(4, 3, 3)
    data_comparison = pd.DataFrame({
        'Token Length': np.concatenate([raw_train_tokens, clean_train_tokens]),
        'Type': ['Raw']*len(raw_train_tokens) + ['Cleaned']*len(clean_train_tokens)
    })
    sns.violinplot(data=data_comparison, x='Type', y='Token Length')
    plt.title('Train: Raw vs Cleaned (Violin)', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(4, 3, 4)
    percentiles = np.arange(0, 101, 5)
    raw_train_pct = np.percentile(raw_train_tokens, percentiles)
    clean_train_pct = np.percentile(clean_train_tokens, percentiles)
    plt.plot(percentiles, raw_train_pct, marker='o', label='Raw', linewidth=2, color='coral')
    plt.plot(percentiles, clean_train_pct, marker='s', label='Cleaned', linewidth=2, color='skyblue')
    plt.xlabel('Percentile', fontsize=12)
    plt.ylabel('Token Length', fontsize=12)
    plt.title('Train: Percentile Comparison', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(4, 3, 5)
    max_lengths = [128, 256, 512, 1024, 2048]
    raw_coverage = [np.sum(raw_train_tokens <= ml) / len(raw_train_tokens) * 100 for ml in max_lengths]
    clean_coverage = [np.sum(clean_train_tokens <= ml) / len(clean_train_tokens) * 100 for ml in max_lengths]
    x = np.arange(len(max_lengths))
    width = 0.35
    plt.bar(x - width/2, raw_coverage, width, label='Raw', edgecolor='black', color='coral')
    plt.bar(x + width/2, clean_coverage, width, label='Cleaned', edgecolor='black', color='skyblue')
    plt.xlabel('Max Token Length', fontsize=12)
    plt.ylabel('Coverage (%)', fontsize=12)
    plt.title('Train: Coverage Comparison', fontsize=14, fontweight='bold')
    plt.xticks(x, max_lengths)
    plt.legend()
    plt.grid(True, alpha=0.3, axis='y')
    
    plt.subplot(4, 3, 6)
    ranges = [(0, 128), (128, 256), (256, 512), (512, 1024), (1024, 2048), (2048, 10000)]
    range_labels = ['0-128', '128-256', '256-512', '512-1024', '1024-2048', '2048+']
    raw_dist = [np.sum((raw_train_tokens >= start) & (raw_train_tokens < end)) for start, end in ranges]
    clean_dist = [np.sum((clean_train_tokens >= start) & (clean_train_tokens < end)) for start, end in ranges]
    x = np.arange(len(range_labels))
    width = 0.35
    plt.bar(x - width/2, raw_dist, width, label='Raw', edgecolor='black', color='coral')
    plt.bar(x + width/2, clean_dist, width, label='Cleaned', edgecolor='black', color='skyblue')
    plt.xlabel('Token Range', fontsize=12)
    plt.ylabel('Count', fontsize=12)
    plt.title('Train: Token Range Distribution', fontsize=14, fontweight='bold')
    plt.xticks(x, range_labels, rotation=45)
    plt.legend()
    plt.grid(True, alpha=0.3, axis='y')
    
    plt.subplot(4, 3, 7)
    reduction = ((raw_train_tokens - clean_train_tokens) / raw_train_tokens * 100)
    reduction = reduction[~np.isnan(reduction) & ~np.isinf(reduction)]
    plt.hist(reduction, bins=50, edgecolor='black', color='green', alpha=0.7)
    plt.xlabel('Token Reduction (%)', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.title(f'Train: Token Reduction Distribution\nMean: {reduction.mean():.1f}%', 
              fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(4, 3, 8)
    raw_cumsum = np.sort(raw_train_tokens)
    raw_cumulative = np.arange(1, len(raw_cumsum) + 1) / len(raw_cumsum) * 100
    clean_cumsum = np.sort(clean_train_tokens)
    clean_cumulative = np.arange(1, len(clean_cumsum) + 1) / len(clean_cumsum) * 100
    plt.plot(raw_cumsum, raw_cumulative, label='Raw', linewidth=2, color='coral')
    plt.plot(clean_cumsum, clean_cumulative, label='Cleaned', linewidth=2, color='skyblue')
    plt.xlabel('Token Length', fontsize=12)
    plt.ylabel('Cumulative Percentage (%)', fontsize=12)
    plt.title('Train: Cumulative Distribution', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(4, 3, 9)
    stats_comparison = {
        'Mean': [raw_train_tokens.mean(), clean_train_tokens.mean()],
        'Median': [np.median(raw_train_tokens), np.median(clean_train_tokens)],
        '90th %ile': [np.percentile(raw_train_tokens, 90), np.percentile(clean_train_tokens, 90)],
        'Max': [raw_train_tokens.max(), clean_train_tokens.max()]
    }
    x = np.arange(len(stats_comparison))
    width = 0.35
    raw_vals = [v[0] for v in stats_comparison.values()]
    clean_vals = [v[1] for v in stats_comparison.values()]
    plt.bar(x - width/2, raw_vals, width, label='Raw', edgecolor='black', color='coral')
    plt.bar(x + width/2, clean_vals, width, label='Cleaned', edgecolor='black', color='skyblue')
    plt.xlabel('Statistic', fontsize=12)
    plt.ylabel('Token Count', fontsize=12)
    plt.title('Train: Statistical Comparison', fontsize=14, fontweight='bold')
    plt.xticks(x, list(stats_comparison.keys()), rotation=45)
    plt.legend()
    plt.grid(True, alpha=0.3, axis='y')
    
    label_col = None
    for col in ['label', 'target', 'class', 'y']:
        if col in train_df.columns:
            label_col = col
            break
    
    if label_col:
        plt.subplot(4, 3, 10)
        train_labels = train_df[label_col].unique()
        for label in sorted(train_labels):
            label_raw = raw_train_tokens[train_df[label_col] == label]
            label_clean = clean_train_tokens[train_df[label_col] == label]
            
            x_pos = [0, 1] if label == sorted(train_labels)[0] else [3, 4]
            means = [label_raw.mean(), label_clean.mean()]
            colors = ['coral', 'skyblue']
            
            plt.bar(x_pos, means, color=colors, edgecolor='black', alpha=0.7)
        
        plt.xticks([0.5, 3.5], [f'Class {l}' for l in sorted(train_labels)])
        plt.ylabel('Mean Token Length', fontsize=12)
        plt.title('Mean Tokens by Class (Red=Raw, Blue=Clean)', fontsize=14, fontweight='bold')
        plt.grid(True, alpha=0.3, axis='y')
        
        plt.subplot(4, 3, 11)
        for label in sorted(train_labels):
            label_tokens = clean_train_tokens[train_df[label_col] == label]
            plt.hist(label_tokens, bins=30, alpha=0.6, label=f'Class {label}', edgecolor='black')
        plt.xlabel('Token Length', fontsize=12)
        plt.ylabel('Frequency', fontsize=12)
        plt.title('Cleaned Token Distribution by Class', fontsize=14, fontweight='bold')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.subplot(4, 3, 12)
        truncation_data = []
        for max_len in [256, 512, 1024]:
            for label in sorted(train_labels):
                label_tokens = clean_train_tokens[train_df[label_col] == label]
                truncated_pct = np.sum(label_tokens > max_len) / len(label_tokens) * 100
                truncation_data.append({
                    'Max Length': max_len,
                    'Class': f'Class {label}',
                    'Truncated %': truncated_pct
                })
        
        df_trunc = pd.DataFrame(truncation_data)
        pivot_trunc = df_trunc.pivot(index='Max Length', columns='Class', values='Truncated %')
        pivot_trunc.plot(kind='bar', ax=plt.gca(), edgecolor='black')
        plt.xlabel('Max Length', fontsize=12)
        plt.ylabel('Truncated (%)', fontsize=12)
        plt.title('Truncation by Class & Max Length', fontsize=14, fontweight='bold')
        plt.legend()
        plt.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('token_analysis_with_preprocessing.png', dpi=300, bbox_inches='tight')
    print("\nVisualization saved: token_analysis_with_preprocessing.png")
    
    return fig

def generate_preprocessing_recommendations(raw_train, raw_test, clean_train, clean_test, train_df, test_df):
    print(f"\n{'='*80}")
    print(f"PREPROCESSING IMPACT & RECOMMENDATIONS")
    print(f"{'='*80}")
    
    print(f"\n1. TOKEN REDUCTION STATISTICS")
    train_reduction = ((raw_train - clean_train) / raw_train * 100)
    test_reduction = ((raw_test - clean_test) / raw_test * 100)
    train_reduction = train_reduction[~np.isnan(train_reduction) & ~np.isinf(train_reduction)]
    test_reduction = test_reduction[~np.isnan(test_reduction) & ~np.isinf(test_reduction)]
    
    print(f"   Train set:")
    print(f"     Average reduction: {train_reduction.mean():.2f}%")
    print(f"     Median reduction: {np.median(train_reduction):.2f}%")
    print(f"     Max reduction: {train_reduction.max():.2f}%")
    print(f"   Test set:")
    print(f"     Average reduction: {test_reduction.mean():.2f}%")
    print(f"     Median reduction: {np.median(test_reduction):.2f}%")
    print(f"     Max reduction: {test_reduction.max():.2f}%")
    
    print(f"\n2. MAX SEQUENCE LENGTH RECOMMENDATIONS")
    print(f"\n   WITHOUT preprocessing (raw code):")
    for coverage in [90, 95, 99]:
        recommended = int(np.percentile(np.concatenate([raw_train, raw_test]), coverage))
        print(f"     {coverage}% coverage: max_length = {recommended}")
    
    print(f"\n   WITH preprocessing (cleaned code):")
    for coverage in [90, 95, 99]:
        recommended = int(np.percentile(np.concatenate([clean_train, clean_test]), coverage))
        print(f"     {coverage}% coverage: max_length = {recommended}")
    
    print(f"\n3. TRUNCATION COMPARISON AT COMMON MAX_LENGTHS")
    for max_len in [256, 512, 1024]:
        raw_trunc = np.sum(raw_train > max_len) / len(raw_train) * 100
        clean_trunc = np.sum(clean_train > max_len) / len(clean_train) * 100
        improvement = raw_trunc - clean_trunc
        print(f"\n   max_length = {max_len}:")
        print(f"     Raw code truncated: {raw_trunc:.2f}%")
        print(f"     Cleaned code truncated: {clean_trunc:.2f}%")
        print(f"     Improvement: {improvement:.2f}% fewer samples truncated")
    
    print(f"\n4. MEMORY & COMPUTATION SAVINGS")
    raw_total = raw_train.sum() + raw_test.sum()
    clean_total = clean_train.sum() + clean_test.sum()
    savings = (raw_total - clean_total) / raw_total * 100
    print(f"   Total token reduction: {savings:.2f}%")
    print(f"   Estimated training time reduction: ~{savings * 0.8:.2f}%")
    print(f"   Estimated memory savings: ~{savings * 0.7:.2f}%")
    
    print(f"\n5. RECOMMENDATION SUMMARY")
    if train_reduction.mean() > 15:
        print(f"   Use code preprocessing")
        print(f"      - Reduces tokens by {train_reduction.mean():.1f}% on average")
        print(f"      - Significantly reduces truncation")
        print(f"      - Maintains semantic information")
    elif train_reduction.mean() > 8:
        print(f"    Consider preprocessing")
        print(f"      - Modest token reduction ({train_reduction.mean():.1f}%)")
        print(f"      - Some benefit for long sequences")
    else:
        print(f"    Preprocessing has minimal impact")
        print(f"      - Only {train_reduction.mean():.1f}% reduction")

print("="*80)
print("VULNERABILITY DETECTION DATASET ANALYSIS WITH PREPROCESSING")
print("="*80)

train_df = load_and_inspect_dataset(train_csv, "TRAINING SET")
test_df = load_and_inspect_dataset(test_csv, "TEST SET")

text_column = None
for col in train_df.columns:
    if train_df[col].dtype == 'object' and col.lower() in ['code', 'func', 'function', 'text', 'source']:
        text_column = col
        break

if text_column is None:
    text_column = train_df.select_dtypes(include=['object']).columns[0]

print(f"\nUsing column '{text_column}' for token analysis...")

print("\n" + "="*80)
print("ANALYZING RAW CODE (without preprocessing)")
print("="*80)
raw_train_tokens = analyze_token_lengths(train_df, "TRAINING SET", text_column, apply_cleaning=False)
raw_test_tokens = analyze_token_lengths(test_df, "TEST SET", text_column, apply_cleaning=False)

print("\n" + "="*80)
print("ANALYZING CLEANED CODE (with preprocessing)")
print("="*80)
clean_train_tokens = analyze_token_lengths(train_df, "TRAINING SET", text_column, apply_cleaning=True)
clean_test_tokens = analyze_token_lengths(test_df, "TEST SET", text_column, apply_cleaning=True)

create_comparison_visualizations(raw_train_tokens, raw_test_tokens, 
                                clean_train_tokens, clean_test_tokens,
                                train_df, test_df)

generate_preprocessing_recommendations(raw_train_tokens, raw_test_tokens,
                                      clean_train_tokens, clean_test_tokens,
                                      train_df, test_df)

print(f"\n{'='*80}")
print("ANALYSIS COMPLETE!")
print(f"{'='*80}")

In [ ]:
import os
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import numpy as np
from scipy import stats
from scipy.stats import ks_2samp, mannwhitneyu, entropy
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

try:
    from transformers import AutoTokenizer
    TOKENIZER_AVAILABLE = True
except (ImportError, AttributeError, Exception):
    TOKENIZER_AVAILABLE = False
    print("Warning: transformers/torch unavailable — falling back to whitespace tokenizer.")

OUT_DIR = 'dataset_analysis_figures'
os.makedirs(OUT_DIR, exist_ok=True)

ORIG_TRAIN_CSV = '/traincodex.csv'
ORIG_TEST_CSV  = '/testcodex.csv'
PROC_TRAIN_CSV = '/trqw.csv'
PROC_TEST_CSV  = '/teqw.csv'

PROC_TEXT_COL  = 'func_cleaned'
ORIG_TEXT_COL  = None
LABEL_COL      = 'label'
MODEL_NAME     = 'microsoft/unixcoder-base'

VULN_LABEL   = 1
BENIGN_LABEL = 0
VULN_NAME    = 'Vulnerable'
BENIGN_NAME  = 'Benign'

COL_OT  = '#E07B39'
COL_OE  = '#C0392B'
COL_PT  = '#5D6D7E'
COL_PE  = '#7F8C8D'

PALETTE_4 = [COL_OT, COL_OE, COL_PT, COL_PE]

LABEL_OT = 'Original–Train'
LABEL_OE = 'Original–Test'
LABEL_PT = 'Processed–Train'
LABEL_PE = 'Processed–Test'

LABELS_4 = [LABEL_OT, LABEL_OE, LABEL_PT, LABEL_PE]

BG         = '#FAFAFA'
GRID_COLOR = '#E8E8E8'
ACCENT     = '#E07B39'
NEUTRAL    = '#7F8C8D'
ORANGE_LIGHT = '#F5CBA7'
GREY_DARK  = '#2C3E50'
GREY_MID   = '#7F8C8D'
GREY_LIGHT = '#BDC3C7'

plt.rcParams.update({
    'figure.facecolor' : BG,
    'axes.facecolor'   : BG,
    'axes.edgecolor'   : '#CCCCCC',
    'axes.labelcolor'  : '#1A1A2E',
    'axes.titlepad'    : 10,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.color'       : GRID_COLOR,
    'grid.linewidth'   : 0.7,
    'xtick.color'      : '#333333',
    'ytick.color'      : '#333333',
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 10,
    'axes.titlesize'   : 11,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 10,
    'legend.frameon'   : True,
    'legend.framealpha': 0.92,
    'legend.edgecolor' : '#CCCCCC',
    'legend.fontsize'  : 8.5,
    'savefig.facecolor': BG,
    'savefig.dpi'      : 300,
    'savefig.bbox'     : 'tight',
})

def save_fig(name):
    path = os.path.join(OUT_DIR, name)
    plt.savefig(path, dpi=300, bbox_inches='tight', facecolor=BG)
    plt.close()
    print(f'Saved: {path}')

def thousands_fmt(x, pos):
    return f'{int(x):,}' if x >= 1000 else f'{int(x)}'

def detect_text_col(df):
    candidates = ['func', 'code', 'function', 'text', 'source', 'content']
    for c in candidates:
        if c in df.columns and df[c].dtype == object:
            return c
    obj_cols = df.select_dtypes(include='object').columns.tolist()
    label_col_candidates = [LABEL_COL, 'target', 'class', 'y']
    for c in obj_cols:
        if c not in label_col_candidates:
            return c
    return obj_cols[0]

def load_all():
    orig_tr = pd.read_csv(ORIG_TRAIN_CSV)
    orig_te = pd.read_csv(ORIG_TEST_CSV)
    proc_tr = pd.read_csv(PROC_TRAIN_CSV)
    proc_te = pd.read_csv(PROC_TEST_CSV)
    global ORIG_TEXT_COL
    if ORIG_TEXT_COL is None:
        ORIG_TEXT_COL = detect_text_col(orig_tr)
    print(f'Original text column detected: "{ORIG_TEXT_COL}"')
    for df in [orig_tr, orig_te]:
        df.dropna(subset=[ORIG_TEXT_COL, LABEL_COL], inplace=True)
        df[LABEL_COL] = df[LABEL_COL].astype(int)
        df.reset_index(drop=True, inplace=True)
    for df in [proc_tr, proc_te]:
        df.dropna(subset=[PROC_TEXT_COL, LABEL_COL], inplace=True)
        df[LABEL_COL] = df[LABEL_COL].astype(int)
        df.reset_index(drop=True, inplace=True)
    return orig_tr, orig_te, proc_tr, proc_te

def tokenize(df, text_col):
    if not TOKENIZER_AVAILABLE:
        return df[text_col].astype(str).apply(lambda x: len(x.split())).values
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    out = []
    codes = df[text_col].astype(str).tolist()
    for i, code in enumerate(codes):
        ids = tokenizer.encode(code, add_special_tokens=True, truncation=False)
        out.append(len(ids))
        if (i + 1) % 2000 == 0:
            print(f'    {i+1}/{len(codes)} tokenized')
    return np.array(out)

def compute_features(df, text_col):
    df = df.copy()
    codes = df[text_col].astype(str)
    df['char_len']         = codes.str.len()
    df['line_count']       = codes.apply(lambda x: len(x.splitlines()))
    df['word_count']       = codes.apply(lambda x: len(x.split()))
    df['unique_words']     = codes.apply(lambda x: len(set(x.lower().split())))
    df['lexical_density']  = df['unique_words'] / df['word_count'].replace(0, np.nan)
    df['avg_line_len']     = df['char_len'] / df['line_count'].replace(0, np.nan)
    df['semicolons']       = codes.str.count(';')
    df['operators']        = codes.apply(lambda x: sum(x.count(op) for op in
                                         ['==', '!=', '<=', '>=', '<', '>', '&&', '||']))
    df['pointer_ops']      = codes.str.count(r'\*') + codes.str.count(r'&')
    df['func_calls_est']   = codes.str.count(r'\w+\s*\(')
    df['string_literals']  = codes.str.count(r'"[^"]*"')
    df['numeric_literals'] = codes.apply(lambda x: len([w for w in x.split()
                                                         if w.replace('.', '', 1).isdigit()]))
    return df

def cohen_d(a, b):
    na, nb = len(a), len(b)
    pooled = np.sqrt(((na - 1) * np.std(a, ddof=1) ** 2 +
                      (nb - 1) * np.std(b, ddof=1) ** 2) / (na + nb - 2))
    return (np.mean(a) - np.mean(b)) / pooled if pooled != 0 else 0.0

def overlap_coeff(a, b, bins=100):
    lo, hi = min(a.min(), b.min()), max(a.max(), b.max())
    edges = np.linspace(lo, hi, bins + 1)
    ha, _ = np.histogram(a, bins=edges, density=True)
    hb, _ = np.histogram(b, bins=edges, density=True)
    return np.sum(np.minimum(ha, hb)) * (edges[1] - edges[0])

def kl_div(a, b, bins=100):
    lo, hi = min(a.min(), b.min()), max(a.max(), b.max())
    edges = np.linspace(lo, hi, bins + 1)
    ha, _ = np.histogram(a, bins=edges, density=True)
    hb, _ = np.histogram(b, bins=edges, density=True)
    ha = ha + 1e-10; hb = hb + 1e-10
    ha /= ha.sum(); hb /= hb.sum()
    return float(entropy(ha, hb))

def draw_table(ax, df_data, title=None):
    ax.axis('off')
    tbl = ax.table(cellText=df_data.values, colLabels=df_data.columns,
                   loc='center', cellLoc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1.05, 1.6)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor('#DDDDDD')
        if r == 0:
            cell.set_facecolor(GREY_DARK)
            cell.set_text_props(color='white', fontweight='bold')
        elif r % 2 == 0:
            cell.set_facecolor('#F0F0F0')
        else:
            cell.set_facecolor(BG)
    if title:
        ax.set_title(title, fontsize=10, fontweight='bold', pad=4)

def print_summary(datasets, token_arrays):
    print('\n' + '=' * 80)
    for (name, df), tok in zip(datasets, token_arrays):
        vuln_n  = (df[LABEL_COL] == VULN_LABEL).sum()
        ben_n   = (df[LABEL_COL] == BENIGN_LABEL).sum()
        ir      = max(vuln_n, ben_n) / min(vuln_n, ben_n) if min(vuln_n, ben_n) > 0 else float('inf')
        print(f'{name:22s}  n={len(df):6,}  vul={vuln_n:6,}  ben={ben_n:6,}  '
              f'IR={ir:.2f}  tok_med={np.median(tok):.0f}  tok_90={np.percentile(tok,90):.0f}')

def fig0_dataset_insights(orig_tr, orig_te, tok_ot, tok_oe):
    C_TRAIN = '#E07B39'
    C_TEST  = '#7F8C8D'
    fig = plt.figure(figsize=(22, 18), facecolor=BG)
    fig.suptitle('Figure 0  ·  Dataset Distribution Insights — Train vs. Test (Original)',
                 fontsize=14, fontweight='bold', y=1.01, color=GREY_DARK)
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.52, wspace=0.40)

    ax0 = fig.add_subplot(gs[0, 0])
    ax0.hist(tok_ot, bins=60, color=C_TRAIN, alpha=0.75, label='Train', edgecolor='white')
    ax0.hist(tok_oe, bins=60, color=C_TEST,  alpha=0.75, label='Test',  edgecolor='white')
    ax0.set_xlabel('Token Length')
    ax0.set_ylabel('Frequency')
    ax0.set_title('Token Length Distribution (Histogram)')
    ax0.legend()
    ax0.xaxis.set_major_formatter(FuncFormatter(thousands_fmt))

    ax1 = fig.add_subplot(gs[0, 1])
    bp = ax1.boxplot([tok_ot, tok_oe], labels=['Train', 'Test'], patch_artist=True,
                     medianprops=dict(color='white', linewidth=2.5),
                     flierprops=dict(marker='.', markersize=2, alpha=0.3, color=GREY_MID),
                     whiskerprops=dict(color=GREY_MID, linewidth=1.4),
                     capprops=dict(color=GREY_MID, linewidth=1.4))
    for patch, c in zip(bp['boxes'], [C_TRAIN, C_TEST]):
        patch.set_facecolor(c)
        patch.set_alpha(0.85)
    ax1.set_ylabel('Token Length')
    ax1.set_title('Token Length Distribution (Box Plot)')
    ax1.yaxis.set_major_formatter(FuncFormatter(thousands_fmt))

    ax2 = fig.add_subplot(gs[0, 2])
    vp = ax2.violinplot([tok_ot, tok_oe], positions=[1, 2], showmedians=True,
                         showextrema=True)
    for body, c in zip(vp['bodies'], [C_TRAIN, C_TEST]):
        body.set_facecolor(c)
        body.set_alpha(0.75)
    vp['cmedians'].set_color('white')
    vp['cmedians'].set_linewidth(2)
    ax2.set_xticks([1, 2])
    ax2.set_xticklabels(['Train', 'Test'])
    ax2.set_xlabel('Dataset')
    ax2.set_ylabel('Token Length')
    ax2.set_title('Token Length Distribution (Violin Plot)')
    ax2.yaxis.set_major_formatter(FuncFormatter(thousands_fmt))

    ax3 = fig.add_subplot(gs[1, 0])
    pcts = np.arange(0, 101, 5)
    ax3.plot(pcts, np.percentile(tok_ot, pcts), marker='o', markersize=4,
             color=C_TRAIN, linewidth=2, label='Train')
    ax3.plot(pcts, np.percentile(tok_oe, pcts), marker='s', markersize=4,
             color=C_TEST, linewidth=2, label='Test')
    ax3.set_xlabel('Percentile')
    ax3.set_ylabel('Token Length')
    ax3.set_title('Token Length Percentile Plot')
    ax3.legend()
    ax3.yaxis.set_major_formatter(FuncFormatter(thousands_fmt))

    ax4 = fig.add_subplot(gs[1, 1])
    thresholds = [128, 256, 512, 1024, 2048]
    x_t = np.arange(len(thresholds))
    w_t = 0.32
    cov_tr = [np.mean(tok_ot <= t) * 100 for t in thresholds]
    cov_te = [np.mean(tok_oe <= t) * 100 for t in thresholds]
    ax4.bar(x_t - w_t/2, cov_tr, w_t, color=C_TRAIN, label='Train', edgecolor='white', alpha=0.88)
    ax4.bar(x_t + w_t/2, cov_te, w_t, color=C_TEST,  label='Test',  edgecolor='white', alpha=0.88)
    for xi, (cv, ct) in enumerate(zip(cov_tr, cov_te)):
        ax4.text(xi - w_t/2, cv + 0.5, f'{cv:.0f}%', ha='center', fontsize=7.5, fontweight='bold')
        ax4.text(xi + w_t/2, ct + 0.5, f'{ct:.0f}%', ha='center', fontsize=7.5, fontweight='bold')
    ax4.set_xticks(x_t)
    ax4.set_xticklabels([str(t) for t in thresholds])
    ax4.set_xlabel('Max Token Length')
    ax4.set_ylabel('Coverage (%)')
    ax4.set_title('Coverage at Different Max Lengths')
    ax4.set_ylim(0, 110)
    ax4.legend()

    ax5 = fig.add_subplot(gs[1, 2])
    bins_range = [0, 128, 256, 512, 1024, 2048, np.inf]
    range_labels = ['0-128', '128-256', '256-512', '512-1024', '1024-2048', '2048+']
    counts_tr = [np.sum((tok_ot >= bins_range[i]) & (tok_ot < bins_range[i+1]))
                 for i in range(len(range_labels))]
    counts_te = [np.sum((tok_oe >= bins_range[i]) & (tok_oe < bins_range[i+1]))
                 for i in range(len(range_labels))]
    x_r = np.arange(len(range_labels))
    w_r = 0.32
    ax5.bar(x_r - w_r/2, counts_tr, w_r, color=C_TRAIN, label='Train', edgecolor='white', alpha=0.88)
    ax5.bar(x_r + w_r/2, counts_te, w_r, color=C_TEST,  label='Test',  edgecolor='white', alpha=0.88)
    ax5.set_xticks(x_r)
    ax5.set_xticklabels(range_labels, fontsize=8.5)
    ax5.set_xlabel('Token Range')
    ax5.set_ylabel('Count')
    ax5.set_title('Token Range Distribution')
    ax5.yaxis.set_major_formatter(FuncFormatter(thousands_fmt))
    ax5.legend()

    ax6 = fig.add_subplot(gs[2, 0])
    sorted_tr = np.sort(tok_ot)
    sorted_te = np.sort(tok_oe)
    ax6.plot(sorted_tr, np.linspace(0, 100, len(sorted_tr)), color=C_TRAIN, linewidth=2, label='Train')
    ax6.plot(sorted_te, np.linspace(0, 100, len(sorted_te)), color=C_TEST,  linewidth=2, label='Test')
    for p, ls in [(25, ':'), (50, '--'), (75, ':'), (90, '-.')]:
        ax6.axhline(p, color=GREY_LIGHT, linestyle=ls, linewidth=0.9, alpha=0.8)
        ax6.text(sorted_tr.max() * 0.02, p + 1, f'{p}th', fontsize=7.5, color=GREY_MID)
    ax6.set_xlabel('Token Length')
    ax6.set_ylabel('Cumulative Percentage (%)')
    ax6.set_title('Cumulative Distribution Function')
    ax6.legend()
    ax6.xaxis.set_major_formatter(FuncFormatter(thousands_fmt))

    ax7 = fig.add_subplot(gs[2, 1])
    mask0_tr = orig_tr[LABEL_COL].values == BENIGN_LABEL
    mask1_tr = orig_tr[LABEL_COL].values == VULN_LABEL
    ax7.hist(tok_ot[mask0_tr], bins=50, color=C_TRAIN, alpha=0.65, label='Class 0', edgecolor='white')
    ax7.hist(tok_ot[mask1_tr], bins=50, color=C_TEST,  alpha=0.65, label='Class 1', edgecolor='white')
    ax7.set_xlabel('Token Length')
    ax7.set_ylabel('Frequency')
    ax7.set_title('Token Distribution by Class (Train)')
    ax7.legend()
    ax7.xaxis.set_major_formatter(FuncFormatter(thousands_fmt))

    ax8 = fig.add_subplot(gs[2, 2])
    classes = [0, 1]
    mean_tr = [tok_ot[orig_tr[LABEL_COL].values == c].mean() for c in classes]
    mean_te = [tok_oe[orig_te[LABEL_COL].values == c].mean() for c in classes]
    x_c = np.arange(len(classes))
    w_c = 0.32
    bars_tr = ax8.bar(x_c - w_c/2, mean_tr, w_c, color=C_TRAIN, label='Train', edgecolor='white', alpha=0.88)
    bars_te = ax8.bar(x_c + w_c/2, mean_te, w_c, color=C_TEST,  label='Test',  edgecolor='white', alpha=0.88)
    for bar in bars_tr:
        ax8.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{bar.get_height():.0f}', ha='center', fontsize=9, fontweight='bold')
    for bar in bars_te:
        ax8.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{bar.get_height():.0f}', ha='center', fontsize=9, fontweight='bold')
    ax8.set_xticks(x_c)
    ax8.set_xticklabels(['Class 0\n(Benign)', 'Class 1\n(Vuln)'])
    ax8.set_xlabel('Class')
    ax8.set_ylabel('Mean Token Length')
    ax8.set_title('Mean Token Length by Class')
    ax8.legend()

    stat_text = (
        f"Train  n={len(tok_ot):,}  |  median={int(np.median(tok_ot))}  "
        f"mean={tok_ot.mean():.0f}  std={tok_ot.std():.0f}  "
        f"p90={int(np.percentile(tok_ot,90))}  max={tok_ot.max()}\n"
        f"Test   n={len(tok_oe):,}  |  median={int(np.median(tok_oe))}  "
        f"mean={tok_oe.mean():.0f}  std={tok_oe.std():.0f}  "
        f"p90={int(np.percentile(tok_oe,90))}  max={tok_oe.max()}"
    )
    fig.text(0.5, -0.01, stat_text, ha='center', fontsize=9.5,
             color=GREY_DARK, style='italic',
             bbox=dict(facecolor=ORANGE_LIGHT, alpha=0.4, edgecolor=ACCENT, boxstyle='round,pad=0.4'))

    save_fig('fig0_dataset_insights.png')

def fig1_class_overview(orig_tr, orig_te, proc_tr, proc_te):
    fig = plt.figure(figsize=(22, 12), facecolor=BG)
    fig.suptitle('Figure 1  ·  Dataset Overview & Class Distribution — All Four Splits',
                 fontsize=14, fontweight='bold', y=1.01, color=GREY_DARK)
    gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.5, wspace=0.38)
    dsets = [(orig_tr, LABEL_OT, COL_OT),
             (orig_te, LABEL_OE, COL_OE),
             (proc_tr, LABEL_PT, COL_PT),
             (proc_te, LABEL_PE, COL_PE)]
    for idx, (df, name, c) in enumerate(dsets):
        ax = fig.add_subplot(gs[0, idx])
        counts = df[LABEL_COL].value_counts().sort_index()
        w, t, at = ax.pie(
            counts.values,
            labels=[BENIGN_NAME, VULN_NAME],
            autopct='%1.1f%%',
            colors=[GREY_LIGHT, c],
            startangle=90,
            pctdistance=0.76,
            wedgeprops=dict(edgecolor='white', linewidth=2))
        for a in at:
            a.set_fontsize(9.5); a.set_fontweight('bold'); a.set_color(GREY_DARK)
        ax.set_title(f'{name}\n({len(df):,} samples)', fontsize=10)
    ax_bar = fig.add_subplot(gs[1, :2])
    split_names = [LABEL_OT, LABEL_OE, LABEL_PT, LABEL_PE]
    vuln_counts   = [(df[LABEL_COL] == VULN_LABEL).sum() for df, _, _ in dsets]
    benign_counts = [(df[LABEL_COL] == BENIGN_LABEL).sum() for df, _, _ in dsets]
    x = np.arange(len(split_names))
    w = 0.36
    ax_bar.bar(x - w/2, benign_counts, w, color=GREY_LIGHT, label=BENIGN_NAME, edgecolor='white')
    ax_bar.bar(x + w/2, vuln_counts,   w, color=PALETTE_4,  label=VULN_NAME,   edgecolor='white')
    ax_bar.set_xticks(x); ax_bar.set_xticklabels(split_names, fontsize=9)
    ax_bar.set_ylabel('Sample Count')
    ax_bar.set_title('Class Count Comparison Across All Splits')
    ax_bar.yaxis.set_major_formatter(FuncFormatter(thousands_fmt))
    for i, (bc, vc) in enumerate(zip(benign_counts, vuln_counts)):
        ax_bar.text(i - w/2, bc + 20, f'{bc:,}', ha='center', fontsize=7.5, color='#444')
        ax_bar.text(i + w/2, vc + 20, f'{vc:,}', ha='center', fontsize=7.5, color='#444')
    p1 = mpatches.Patch(color=GREY_LIGHT, label=BENIGN_NAME)
    ax_bar.legend(handles=[p1] + [mpatches.Patch(color=c, label=n)
                                   for _, n, c in dsets], fontsize=8)
    ax_stack = fig.add_subplot(gs[1, 2])
    vuln_pct   = [(df[LABEL_COL] == VULN_LABEL).mean() * 100 for df, _, _ in dsets]
    benign_pct = [100 - v for v in vuln_pct]
    x2 = np.arange(len(split_names))
    ax_stack.bar(x2, benign_pct, color=GREY_LIGHT, edgecolor='white')
    ax_stack.bar(x2, vuln_pct, bottom=benign_pct, color=PALETTE_4, edgecolor='white')
    ax_stack.set_xticks(x2); ax_stack.set_xticklabels(split_names, fontsize=8, rotation=15)
    ax_stack.set_ylabel('Proportion (%)')
    ax_stack.set_ylim(0, 115)
    ax_stack.set_title('Stacked Class Proportion')
    for xi, (b, v) in enumerate(zip(benign_pct, vuln_pct)):
        ax_stack.text(xi, b / 2,     f'{b:.1f}%', ha='center', va='center',
                      color='white', fontweight='bold', fontsize=9)
        ax_stack.text(xi, b + v / 2, f'{v:.1f}%', ha='center', va='center',
                      color='white', fontweight='bold', fontsize=9)
    ax_ir = fig.add_subplot(gs[1, 3])
    irs = [max(vc, bc) / min(vc, bc) for vc, bc in zip(vuln_counts, benign_counts)]
    bars = ax_ir.bar(split_names, irs, color=PALETTE_4, edgecolor='white', width=0.55)
    ax_ir.axhline(1.0, color=GREY_MID, linestyle='--', linewidth=1.2, label='Balanced IR=1')
    ax_ir.set_ylabel('Imbalance Ratio (IR)')
    ax_ir.set_title('Class Imbalance Ratio')
    ax_ir.set_xticklabels(split_names, fontsize=8, rotation=15)
    for bar, val in zip(bars, irs):
        ax_ir.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                   f'{val:.2f}', ha='center', fontsize=10, fontweight='bold')
    ax_ir.legend(fontsize=8)
    save_fig('fig1_class_overview.png')

def fig2_token_overview(tok_ot, tok_oe, tok_pt, tok_pe):
    tok_all = [tok_ot, tok_oe, tok_pt, tok_pe]
    fig = plt.figure(figsize=(22, 16), facecolor=BG)
    fig.suptitle('Figure 2  ·  Token Length Distribution — All Four Splits',
                 fontsize=14, fontweight='bold', y=1.01, color=GREY_DARK)
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.38)
    ax0 = fig.add_subplot(gs[0, 0])
    for tok, name, c in zip(tok_all, LABELS_4, PALETTE_4):
        ax0.hist(tok, bins=60, alpha=0.45, color=c, label=name, density=False, edgecolor='none')
        ax0.axvline(np.median(tok), color=c, linestyle='--', linewidth=1.4)
    ax0.set_xlabel('Token Length'); ax0.set_ylabel('Frequency')
    ax0.set_title('Histogram (dashed = median)'); ax0.legend()
    ax0.xaxis.set_major_formatter(FuncFormatter(thousands_fmt))
    ax1 = fig.add_subplot(gs[0, 1])
    for tok, name, c in zip(tok_all, LABELS_4, PALETTE_4):
        ax1.hist(tok, bins=60, alpha=0.35, color=c, density=True, edgecolor='none')
        kde = stats.gaussian_kde(tok)
        x_ = np.linspace(tok.min(), tok.max(), 400)
        ax1.plot(x_, kde(x_), color=c, linewidth=2, label=name)
    ax1.set_xlabel('Token Length'); ax1.set_ylabel('Density')
    ax1.set_title('KDE Overlay'); ax1.legend()
    ax2 = fig.add_subplot(gs[0, 2])
    for tok, name, c in zip(tok_all, LABELS_4, PALETTE_4):
        sorted_t = np.sort(tok)
        ax2.plot(sorted_t, np.linspace(0, 100, len(sorted_t)), color=c, linewidth=2, label=name)
    for p, ls in [(50, '--'), (90, ':'), (95, '-.')]:
        ax2.axhline(p, color='gray', linestyle=ls, linewidth=0.8, alpha=0.6)
        ax2.text(max(tok.max() for tok in tok_all) * 0.02, p + 0.6,
                 f'{p}th', fontsize=7.5, color='gray')
    ax2.set_xlabel('Token Length'); ax2.set_ylabel('Cumulative (%)')
    ax2.set_title('Empirical CDF'); ax2.legend()
    ax3 = fig.add_subplot(gs[1, 0])
    bp = ax3.boxplot(tok_all, labels=LABELS_4, patch_artist=True,
                     medianprops=dict(color='white', linewidth=2),
                     flierprops=dict(marker='.', markersize=2, alpha=0.2))
    for patch, c in zip(bp['boxes'], PALETTE_4):
        patch.set_facecolor(c); patch.set_alpha(0.7)
    ax3.set_ylabel('Token Length'); ax3.set_title('Box Plot Comparison')
    ax3.set_xticklabels(LABELS_4, fontsize=8, rotation=12)
    ax3.yaxis.set_major_formatter(FuncFormatter(thousands_fmt))
    ax4 = fig.add_subplot(gs[1, 1])
    vp = ax4.violinplot(tok_all, positions=range(4), showmedians=True)
    for body, c in zip(vp['bodies'], PALETTE_4):
        body.set_facecolor(c); body.set_alpha(0.65)
    ax4.set_xticks(range(4)); ax4.set_xticklabels(LABELS_4, fontsize=8, rotation=12)
    ax4.set_ylabel('Token Length'); ax4.set_title('Violin Plot')
    ax4.yaxis.set_major_formatter(FuncFormatter(thousands_fmt))
    ax5 = fig.add_subplot(gs[1, 2])
    pcts = np.arange(5, 100, 5)
    for tok, name, c in zip(tok_all, LABELS_4, PALETTE_4):
        ax5.plot(pcts, np.percentile(tok, pcts), marker='o', markersize=3,
                 linewidth=1.8, color=c, label=name)
    ax5.set_xlabel('Percentile'); ax5.set_ylabel('Token Length')
    ax5.set_title('Percentile Profile'); ax5.legend()
    ax6 = fig.add_subplot(gs[2, 0])
    thresholds = [64, 128, 256, 384, 512, 768, 1024, 1536, 2048]
    for tok, name, c in zip(tok_all, LABELS_4, PALETTE_4):
        cov = [np.mean(tok <= t) * 100 for t in thresholds]
        ax6.plot(thresholds, cov, marker='o', linewidth=2, color=c, label=name)
    for t in [256, 512, 1024]:
        ax6.axvline(t, color='gray', linestyle='--', linewidth=0.9, alpha=0.5)
    ax6.set_xlabel('Max Sequence Length'); ax6.set_ylabel('Coverage (%)')
    ax6.set_title('Coverage vs. Max Sequence Length')
    ax6.legend(); ax6.set_ylim(0, 105)
    ax7 = fig.add_subplot(gs[2, 1])
    thresholds2 = [128, 256, 512, 1024, 2048]
    x = np.arange(len(thresholds2)); w = 0.18
    for i, (tok, name, c) in enumerate(zip(tok_all, LABELS_4, PALETTE_4)):
        trunc = [np.mean(tok > t) * 100 for t in thresholds2]
        ax7.bar(x + (i - 1.5) * w, trunc, w, color=c, label=name, edgecolor='white')
    ax7.set_xticks(x); ax7.set_xticklabels([str(t) for t in thresholds2])
    ax7.set_xlabel('Threshold'); ax7.set_ylabel('Truncated (%)')
    ax7.set_title('Truncation Rate at Common Thresholds')
    ax7.legend(fontsize=8)
    ax8 = fig.add_subplot(gs[2, 2])
    stat_rows = []
    for tok, name in zip(tok_all, LABELS_4):
        stat_rows.append([name, f'{tok.min()}', f'{int(np.median(tok))}',
                          f'{tok.mean():.1f}', f'{tok.std():.1f}',
                          f'{int(np.percentile(tok, 90))}',
                          f'{int(np.percentile(tok, 99))}', f'{tok.max()}'])
    sdf = pd.DataFrame(stat_rows, columns=['Split', 'Min', 'Median', 'Mean',
                                            'Std', 'P90', 'P99', 'Max'])
    draw_table(ax8, sdf, 'Token Length Statistics Summary')
    save_fig('fig2_token_overview.png')

def fig3_preprocessing_impact_tokens(tok_ot, tok_oe, tok_pt, tok_pe):
    fig = plt.figure(figsize=(22, 14), facecolor=BG)
    fig.suptitle('Figure 3  ·  Preprocessing Impact on Token Length (Original → Processed)',
                 fontsize=14, fontweight='bold', y=1.01, color=GREY_DARK)
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.4)
    ax0 = fig.add_subplot(gs[0, 0])
    for tok, name, c, ls in [(tok_ot, LABEL_OT, COL_OT, '-'),
                               (tok_pt, LABEL_PT, COL_PT, '--')]:
        ax0.hist(tok, bins=60, alpha=0.5, color=c, density=True, edgecolor='none')
        kde = stats.gaussian_kde(tok)
        x_ = np.linspace(tok.min(), tok.max(), 400)
        ax0.plot(x_, kde(x_), color=c, linewidth=2.2, linestyle=ls, label=name)
    ax0.set_xlabel('Token Length'); ax0.set_ylabel('Density')
    ax0.set_title('Train Split — Original vs. Processed'); ax0.legend()
    ax1 = fig.add_subplot(gs[0, 1])
    for tok, name, c, ls in [(tok_oe, LABEL_OE, COL_OE, '-'),
                               (tok_pe, LABEL_PE, COL_PE, '--')]:
        ax1.hist(tok, bins=60, alpha=0.5, color=c, density=True, edgecolor='none')
        kde = stats.gaussian_kde(tok)
        x_ = np.linspace(tok.min(), tok.max(), 400)
        ax1.plot(x_, kde(x_), color=c, linewidth=2.2, linestyle=ls, label=name)
    ax1.set_xlabel('Token Length'); ax1.set_ylabel('Density')
    ax1.set_title('Test Split — Original vs. Processed'); ax1.legend()
    ax2 = fig.add_subplot(gs[0, 2])
    for orig_tok, proc_tok, name, c in [
        (tok_ot, tok_pt, 'Train', COL_OT),
        (tok_oe, tok_pe, 'Test',  COL_OE)]:
        sorted_o = np.sort(orig_tok)
        sorted_p = np.sort(proc_tok)
        ax2.plot(sorted_o, np.linspace(0, 100, len(sorted_o)),
                 color=c, linewidth=2, label=f'Original–{name}')
        ax2.plot(sorted_p, np.linspace(0, 100, len(sorted_p)),
                 color=c, linewidth=2, linestyle='--', label=f'Processed–{name}')
    for p, ls in [(50, ':'), (90, ':'), (95, ':')]:
        ax2.axhline(p, color='gray', linestyle=ls, linewidth=0.7, alpha=0.5)
    ax2.set_xlabel('Token Length'); ax2.set_ylabel('Cumulative (%)')
    ax2.set_title('CDF Shift: Original → Processed'); ax2.legend(fontsize=7.5)
    ax3 = fig.add_subplot(gs[1, 0])
    thresholds = [64, 128, 256, 384, 512, 768, 1024, 1536, 2048]
    for orig_tok, proc_tok, name, co, cp in [
        (tok_ot, tok_pt, 'Train', COL_OT, COL_PT),
        (tok_oe, tok_pe, 'Test',  COL_OE, COL_PE)]:
        cov_o = [np.mean(orig_tok <= t) * 100 for t in thresholds]
        cov_p = [np.mean(proc_tok <= t) * 100 for t in thresholds]
        ax3.plot(thresholds, cov_o, color=co, linewidth=2, label=f'Original–{name}')
        ax3.plot(thresholds, cov_p, color=cp, linewidth=2, linestyle='--',
                 label=f'Processed–{name}')
        ax3.fill_between(thresholds, cov_o, cov_p, alpha=0.08, color=co)
    ax3.set_xlabel('Max Sequence Length'); ax3.set_ylabel('Coverage (%)')
    ax3.set_title('Coverage Gain after Preprocessing')
    ax3.legend(fontsize=7.5); ax3.set_ylim(0, 105)
    ax4 = fig.add_subplot(gs[1, 1])
    thresholds2 = [128, 256, 512, 1024, 2048]
    x = np.arange(len(thresholds2)); w = 0.2
    pairs = [(tok_ot, tok_pt, COL_OT, COL_PT, 'Train'),
             (tok_oe, tok_pe, COL_OE, COL_PE, 'Test')]
    for i, (orig_tok, proc_tok, co, cp, sp_name) in enumerate(pairs):
        trunc_o = [np.mean(orig_tok > t) * 100 for t in thresholds2]
        trunc_p = [np.mean(proc_tok > t) * 100 for t in thresholds2]
        ax4.bar(x + (2*i - 1.5) * w, trunc_o, w, color=co, label=f'Original–{sp_name}',
                edgecolor='white')
        ax4.bar(x + (2*i - 0.5) * w, trunc_p, w, color=cp, label=f'Processed–{sp_name}',
                edgecolor='white')
    ax4.set_xticks(x); ax4.set_xticklabels([str(t) for t in thresholds2])
    ax4.set_xlabel('Threshold'); ax4.set_ylabel('Truncated Samples (%)')
    ax4.set_title('Truncation Rate Reduction'); ax4.legend(fontsize=7.5)
    ax5 = fig.add_subplot(gs[1, 2])
    pairs_info = [
        ('Train: Orig→Proc', tok_ot, tok_pt, COL_OT, COL_PT),
        ('Test:  Orig→Proc', tok_oe, tok_pe, COL_OE, COL_PE),
    ]
    rows = []
    for label, o_tok, p_tok, co, cp in pairs_info:
        ks_s, ks_p = ks_2samp(o_tok, p_tok)
        d = abs(cohen_d(o_tok, p_tok))
        oc = overlap_coeff(o_tok, p_tok)
        kl = kl_div(o_tok, p_tok)
        med_red  = (np.median(o_tok) - np.median(p_tok)) / np.median(o_tok) * 100
        mean_red = (o_tok.mean() - p_tok.mean()) / o_tok.mean() * 100
        rows.append([label, f'{med_red:.1f}%', f'{mean_red:.1f}%',
                     f'{ks_s:.4f}', f'{ks_p:.2e}', f"{d:.4f}", f'{oc:.4f}', f'{kl:.4f}'])
    sdf = pd.DataFrame(rows, columns=['Comparison', 'Median↓%', 'Mean↓%',
                                       'KS Stat', 'KS p', "Cohen's|d|", 'Overlap', 'KL Div'])
    draw_table(ax5, sdf, 'Preprocessing Impact Statistics')
    save_fig('fig3_preprocessing_impact_tokens.png')

def fig4_class_stratified(orig_tr, orig_te, proc_tr, proc_te,
                           tok_ot, tok_oe, tok_pt, tok_pe):
    fig = plt.figure(figsize=(22, 14), facecolor=BG)
    fig.suptitle('Figure 4  ·  Class-Stratified Token Length Analysis',
                 fontsize=14, fontweight='bold', y=1.01, color=GREY_DARK)
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.4)
    split_data = [
        (orig_tr, tok_ot, LABEL_OT, COL_OT),
        (orig_te, tok_oe, LABEL_OE, COL_OE),
        (proc_tr, tok_pt, LABEL_PT, COL_PT),
        (proc_te, tok_pe, LABEL_PE, COL_PE),
    ]
    ax0 = fig.add_subplot(gs[0, 0])
    for df, tok, name, c in split_data:
        for lbl, cls, ls in [(VULN_LABEL, VULN_NAME, '-'),
                              (BENIGN_LABEL, BENIGN_NAME, '--')]:
            sub = tok[df[LABEL_COL].values == lbl]
            if len(sub) < 5:
                continue
            kde = stats.gaussian_kde(sub)
            x_ = np.linspace(sub.min(), sub.max(), 300)
            ax0.plot(x_, kde(x_), color=c, linestyle=ls, linewidth=1.5,
                     label=f'{name[:4]}/{cls[:3]}', alpha=0.85)
    ax0.set_xlabel('Token Length'); ax0.set_ylabel('Density')
    ax0.set_title('Class-Stratified KDE — All Splits\n(solid=Vuln, dash=Benign)')
    ax0.legend(fontsize=7)
    ax1 = fig.add_subplot(gs[0, 1])
    box_data, box_labels = [], []
    for df, tok, name, c in split_data:
        for lbl, cls in [(VULN_LABEL, VULN_NAME), (BENIGN_LABEL, BENIGN_NAME)]:
            box_data.append(tok[df[LABEL_COL].values == lbl])
            box_labels.append(f'{name[:6]}\n{cls[:3]}')
    bp = ax1.boxplot(box_data, labels=box_labels, patch_artist=True,
                     medianprops=dict(color='white', linewidth=1.8),
                     flierprops=dict(marker='.', markersize=1.5, alpha=0.2))
    colors_box = [c for c in PALETTE_4 for _ in range(2)]
    alphas_box  = [0.85, 0.45] * 4
    for patch, c, a in zip(bp['boxes'], colors_box, alphas_box):
        patch.set_facecolor(c); patch.set_alpha(a)
    ax1.set_ylabel('Token Length'); ax1.set_title('Box Plot: Class × Split')
    ax1.tick_params(axis='x', labelsize=7.5)
    ax1.yaxis.set_major_formatter(FuncFormatter(thousands_fmt))
    ax2 = fig.add_subplot(gs[0, 2])
    pcts = np.arange(5, 100, 5)
    for df, tok, name, c in split_data:
        vuln_tok   = tok[df[LABEL_COL].values == VULN_LABEL]
        benign_tok = tok[df[LABEL_COL].values == BENIGN_LABEL]
        ax2.plot(pcts, np.percentile(vuln_tok,  pcts), color=c, linestyle='-',
                 linewidth=1.6, label=f'{name[:6]} {VULN_NAME[:3]}')
        ax2.plot(pcts, np.percentile(benign_tok, pcts), color=c, linestyle='--',
                 linewidth=1.4, alpha=0.75)
    ax2.set_xlabel('Percentile'); ax2.set_ylabel('Token Length')
    ax2.set_title('Percentile Profiles by Class × Split\n(solid=Vuln, dash=Benign)')
    ax2.legend(fontsize=7)
    ax3 = fig.add_subplot(gs[1, 0])
    thresholds2 = [256, 512, 1024]
    x = np.arange(len(split_data)); w = 0.22
    for ti, t in enumerate(thresholds2):
        trunc_vuln   = [np.mean(tok[df[LABEL_COL].values == VULN_LABEL]  > t)*100
                        for df, tok, _, _ in split_data]
        trunc_benign = [np.mean(tok[df[LABEL_COL].values == BENIGN_LABEL] > t)*100
                        for df, tok, _, _ in split_data]
        ax3.bar(x + (ti-1)*w, trunc_vuln, w, color=PALETTE_4,
                alpha=0.85, edgecolor='white',
                label=f'Vuln @{t}' if ti == 0 else f'@{t}')
        ax3.bar(x + (ti-1)*w, trunc_benign, w, color=PALETTE_4,
                alpha=0.35, edgecolor='white', hatch='//')
    ax3.set_xticks(x); ax3.set_xticklabels(LABELS_4, fontsize=8, rotation=12)
    ax3.set_ylabel('Truncated (%)'); ax3.set_title('Truncation by Class × Split × Threshold')
    vuln_p = mpatches.Patch(facecolor='gray', label='Vuln (solid)')
    ben_p  = mpatches.Patch(facecolor='gray', alpha=0.35, hatch='//', label='Benign (hatch)')
    ax3.legend(handles=[vuln_p, ben_p], fontsize=8)
    rows = []
    for df, tok, name, c in split_data:
        v = tok[df[LABEL_COL].values == VULN_LABEL]
        b = tok[df[LABEL_COL].values == BENIGN_LABEL]
        ks_s, ks_p = ks_2samp(v, b)
        d = abs(cohen_d(v, b))
        oc = overlap_coeff(v, b)
        rows.append([name, f'{v.mean():.0f}', f'{b.mean():.0f}',
                     f'{ks_s:.4f}', f'{ks_p:.2e}', f'{d:.4f}', f'{oc:.4f}'])
    sdf = pd.DataFrame(rows, columns=['Split', 'Vuln Mean', 'Ben Mean',
                                       'KS', 'KS p', "Cohen's|d|", 'Overlap'])
    ax4 = fig.add_subplot(gs[1, 1:])
    draw_table(ax4, sdf, 'Class Separation Statistics per Split')
    save_fig('fig4_class_stratified_tokens.png')

def fig5_code_complexity(orig_tr_f, orig_te_f, proc_tr_f, proc_te_f):
    features = ['char_len', 'line_count', 'word_count', 'lexical_density',
                'avg_line_len', 'semicolons', 'operators', 'pointer_ops', 'func_calls_est']
    flabels  = ['Char Length', 'Line Count', 'Word Count', 'Lexical Density',
                'Avg Line Len', 'Semicolons', 'Operators', 'Pointer Ops', 'Func Calls']
    fig = plt.figure(figsize=(22, 18), facecolor=BG)
    fig.suptitle('Figure 5  ·  Code Structural Feature Distributions — All Four Splits',
                 fontsize=14, fontweight='bold', y=1.01, color=GREY_DARK)
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.52, wspace=0.4)
    all_feat = [(orig_tr_f, LABEL_OT, COL_OT),
                (orig_te_f, LABEL_OE, COL_OE),
                (proc_tr_f, LABEL_PT, COL_PT),
                (proc_te_f, LABEL_PE, COL_PE)]
    for idx, (feat, flabel) in enumerate(zip(features, flabels)):
        row, col = divmod(idx, 3)
        ax = fig.add_subplot(gs[row, col])
        for df, name, c in all_feat:
            vals = df[feat].dropna().values
            hi = np.percentile(vals, 99)
            vals = vals[vals <= hi]
            ax.hist(vals, bins=40, alpha=0.35, color=c, density=True, edgecolor='none')
            try:
                kde = stats.gaussian_kde(vals)
                x_ = np.linspace(vals.min(), vals.max(), 300)
                ax.plot(x_, kde(x_), color=c, linewidth=1.8, label=name)
            except Exception:
                pass
        ax.set_xlabel(flabel, fontsize=9); ax.set_ylabel('Density', fontsize=9)
        ax.set_title(flabel)
        if idx == 0:
            ax.legend(fontsize=7.5)
    save_fig('fig5_code_complexity.png')

def fig6_preprocessing_impact_features(orig_tr_f, orig_te_f, proc_tr_f, proc_te_f):
    features = ['char_len', 'line_count', 'word_count', 'lexical_density',
                'semicolons', 'operators', 'pointer_ops', 'func_calls_est']
    flabels  = ['Char Length', 'Line Count', 'Word Count', 'Lexical Density',
                'Semicolons', 'Operators', 'Pointer Ops', 'Func Calls']
    fig = plt.figure(figsize=(22, 16), facecolor=BG)
    fig.suptitle('Figure 6  ·  Preprocessing Impact on Code Structural Features (Original → Processed)',
                 fontsize=14, fontweight='bold', y=1.01, color=GREY_DARK)
    gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.55, wspace=0.42)
    pairs = [
        (orig_tr_f, proc_tr_f, 'Train', COL_OT, COL_PT),
        (orig_te_f, proc_te_f, 'Test',  COL_OE, COL_PE),
    ]
    for idx, (feat, flabel) in enumerate(zip(features, flabels)):
        row, col = divmod(idx, 4)
        ax = fig.add_subplot(gs[row, col])
        for orig_f, proc_f, sp_name, co, cp in pairs:
            for df, name, c, ls in [(orig_f, f'Orig–{sp_name}', co, '-'),
                                     (proc_f, f'Proc–{sp_name}', cp, '--')]:
                vals = df[feat].dropna().values
                hi = np.percentile(vals, 99)
                vals = vals[vals <= hi]
                try:
                    kde = stats.gaussian_kde(vals)
                    x_ = np.linspace(vals.min(), vals.max(), 300)
                    ax.plot(x_, kde(x_), color=c, linewidth=1.8, linestyle=ls, label=name)
                except Exception:
                    pass
        ax.set_xlabel(flabel, fontsize=8.5); ax.set_ylabel('Density', fontsize=8.5)
        ax.set_title(flabel)
        if idx == 0:
            ax.legend(fontsize=7)
    save_fig('fig6_preprocessing_impact_features.png')

def fig7_class_complexity_contrast(orig_tr_f, proc_tr_f):
    features = ['char_len', 'line_count', 'word_count', 'lexical_density',
                'semicolons', 'operators', 'pointer_ops', 'func_calls_est']
    flabels  = ['Char Length', 'Line Count', 'Word Count', 'Lexical Density',
                'Semicolons', 'Operators', 'Pointer Ops', 'Func Calls']
    fig = plt.figure(figsize=(22, 16), facecolor=BG)
    fig.suptitle('Figure 7  ·  Class-Stratified Structural Features (Original vs. Processed — Train Split)',
                 fontsize=14, fontweight='bold', y=1.01, color=GREY_DARK)
    gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.55, wspace=0.42)
    for idx, (feat, flabel) in enumerate(zip(features, flabels)):
        row, col = divmod(idx, 4)
        ax = fig.add_subplot(gs[row, col])
        for df, sp_co, sp_cp, sp_name in [
            (orig_tr_f, COL_OT, COL_OT, 'Orig'),
            (proc_tr_f, COL_PT, COL_PT, 'Proc')]:
            for lbl, cls, c, ls in [
                (VULN_LABEL,   VULN_NAME,   sp_co, '-'),
                (BENIGN_LABEL, BENIGN_NAME, sp_co, '--')]:
                vals = df[df[LABEL_COL] == lbl][feat].dropna().values
                hi = np.percentile(vals, 99)
                vals = vals[vals <= hi]
                alpha = 0.9 if sp_name == 'Orig' else 0.55
                try:
                    kde = stats.gaussian_kde(vals)
                    x_ = np.linspace(vals.min(), vals.max(), 300)
                    ax.plot(x_, kde(x_), color=c, linestyle=ls, linewidth=1.8,
                            label=f'{sp_name}/{cls[:3]}', alpha=alpha)
                except Exception:
                    pass
        ax.set_xlabel(flabel, fontsize=8.5); ax.set_ylabel('Density', fontsize=8.5)
        ax.set_title(flabel)
        if idx == 0:
            ax.legend(fontsize=7)
    save_fig('fig7_class_complexity_contrast.png')

def fig8_statistical_significance(orig_tr_f, orig_te_f, proc_tr_f, proc_te_f):
    features = ['char_len', 'line_count', 'word_count', 'lexical_density',
                'semicolons', 'operators', 'pointer_ops', 'func_calls_est',
                'avg_line_len', 'string_literals', 'numeric_literals']
    flabels  = ['Char Len', 'Line Cnt', 'Word Cnt', 'Lex Density',
                'Semicolons', 'Operators', 'Ptr Ops', 'Func Calls',
                'Avg Line', 'Str Lit', 'Num Lit']
    split_feats = [(orig_tr_f, LABEL_OT), (orig_te_f, LABEL_OE),
                   (proc_tr_f, LABEL_PT), (proc_te_f, LABEL_PE)]
    results = []
    for df, sp_name in split_feats:
        for feat, fl in zip(features, flabels):
            v = df[df[LABEL_COL] == VULN_LABEL][feat].dropna().values
            b = df[df[LABEL_COL] == BENIGN_LABEL][feat].dropna().values
            if len(v) < 5 or len(b) < 5:
                continue
            _, p = mannwhitneyu(v, b, alternative='two-sided')
            d = abs(cohen_d(v, b))
            oc = overlap_coeff(v, b)
            results.append({'Feature': fl, 'Split': sp_name,
                             'Cohen_d': d, 'p_value': p,
                             'Overlap': oc, 'neg_log_p': -np.log10(p + 1e-300)})
    rdf = pd.DataFrame(results)
    fig = plt.figure(figsize=(22, 14), facecolor=BG)
    fig.suptitle('Figure 8  ·  Statistical Significance & Effect Size — Vulnerable vs. Benign',
                 fontsize=14, fontweight='bold', y=1.01, color=GREY_DARK)
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.5, wspace=0.38)
    ax0 = fig.add_subplot(gs[0, 0])
    for sp_name, c, mk in zip(LABELS_4, PALETTE_4, ['o', 's', '^', 'D']):
        sub = rdf[rdf['Split'] == sp_name]
        ax0.scatter(sub['Cohen_d'], sub['neg_log_p'], color=c, marker=mk,
                    s=70, alpha=0.85, label=sp_name, edgecolors='white', linewidth=0.6)
        for _, row in sub.iterrows():
            ax0.text(row['Cohen_d'] + 0.004, row['neg_log_p'] + 0.25,
                     row['Feature'], fontsize=6.5, color='#333333')
    ax0.axhline(-np.log10(0.05), color=ACCENT, linestyle='--', linewidth=1.3, label='p=0.05')
    ax0.axvline(0.2, color='gray', linestyle=':', linewidth=1, alpha=0.6)
    ax0.axvline(0.5, color='gray', linestyle=':', linewidth=1, alpha=0.6)
    ax0.set_xlabel("Cohen's |d|"); ax0.set_ylabel('-log₁₀(p)')
    ax0.set_title('Volcano Plot — Effect Size vs. Significance'); ax0.legend(fontsize=8)
    ax1 = fig.add_subplot(gs[0, 1])
    pivot_d = rdf.pivot_table(index='Feature', columns='Split', values='Cohen_d')
    x = np.arange(len(pivot_d)); w = 0.18
    for i, (sp_name, c) in enumerate(zip(LABELS_4, PALETTE_4)):
        if sp_name in pivot_d.columns:
            ax1.bar(x + (i - 1.5) * w, pivot_d[sp_name], w, color=c,
                    label=sp_name, edgecolor='white', alpha=0.85)
    for thresh, ls, label in [(0.2, '--', 'Small'), (0.5, '--', 'Medium'), (0.8, '--', 'Large')]:
        ax1.axhline(thresh, color='gray', linestyle=ls, linewidth=0.9, alpha=0.7, label=label)
    ax1.set_xticks(x); ax1.set_xticklabels(pivot_d.index, rotation=40, ha='right', fontsize=8)
    ax1.set_ylabel("Cohen's |d|"); ax1.set_title("Effect Size per Feature × Split")
    ax1.legend(fontsize=7.5)
    ax2 = fig.add_subplot(gs[1, 0])
    pivot_p = rdf.pivot_table(index='Feature', columns='Split', values='neg_log_p')
    for i, (sp_name, c) in enumerate(zip(LABELS_4, PALETTE_4)):
        if sp_name in pivot_p.columns:
            ax2.bar(x + (i - 1.5) * w, pivot_p[sp_name], w, color=c,
                    label=sp_name, edgecolor='white', alpha=0.85)
    ax2.axhline(-np.log10(0.05), color=ACCENT, linestyle='--', linewidth=1.5, label='p=0.05')
    ax2.set_xticks(x); ax2.set_xticklabels(pivot_p.index, rotation=40, ha='right', fontsize=8)
    ax2.set_ylabel('-log₁₀(p)'); ax2.set_title('Significance (Mann–Whitney U)')
    ax2.legend(fontsize=7.5)
    ax3 = fig.add_subplot(gs[1, 1])
    pivot_oc = rdf.pivot_table(index='Feature', columns='Split', values='Overlap')
    for i, (sp_name, c) in enumerate(zip(LABELS_4, PALETTE_4)):
        if sp_name in pivot_oc.columns:
            ax3.bar(x + (i - 1.5) * w, pivot_oc[sp_name], w, color=c,
                    label=sp_name, edgecolor='white', alpha=0.85)
    ax3.set_xticks(x); ax3.set_xticklabels(pivot_oc.index, rotation=40, ha='right', fontsize=8)
    ax3.set_ylabel('Overlap Coefficient')
    ax3.set_title('Class Distribution Overlap (↓ = more separable)')
    ax3.legend(fontsize=7.5); ax3.set_ylim(0, 1.05)
    save_fig('fig8_statistical_significance.png')

def fig9_correlation_heatmaps(orig_tr_f, proc_tr_f):
    numeric_feats = ['char_len', 'line_count', 'word_count', 'lexical_density',
                     'avg_line_len', 'semicolons', 'operators', 'pointer_ops',
                     'func_calls_est', 'string_literals', 'numeric_literals']
    feat_short = ['CL', 'LC', 'WC', 'LD', 'AL', 'SC', 'OP', 'PO', 'FC', 'SL', 'NL']
    cmap = LinearSegmentedColormap.from_list('og', ['#2C3E50', 'white', '#E07B39'], N=256)
    fig, axes = plt.subplots(2, 3, figsize=(22, 13), facecolor=BG)
    fig.suptitle('Figure 9  ·  Feature Correlation Heatmaps — Original vs. Processed (All / Vulnerable / Benign)',
                 fontsize=14, fontweight='bold', y=1.01, color=GREY_DARK)
    panels = [
        (axes[0, 0], orig_tr_f,                               f'{LABEL_OT} — All'),
        (axes[0, 1], orig_tr_f[orig_tr_f[LABEL_COL]==VULN_LABEL],   f'{LABEL_OT} — {VULN_NAME}'),
        (axes[0, 2], orig_tr_f[orig_tr_f[LABEL_COL]==BENIGN_LABEL], f'{LABEL_OT} — {BENIGN_NAME}'),
        (axes[1, 0], proc_tr_f,                               f'{LABEL_PT} — All'),
        (axes[1, 1], proc_tr_f[proc_tr_f[LABEL_COL]==VULN_LABEL],   f'{LABEL_PT} — {VULN_NAME}'),
        (axes[1, 2], proc_tr_f[proc_tr_f[LABEL_COL]==BENIGN_LABEL], f'{LABEL_PT} — {BENIGN_NAME}'),
    ]
    for ax, df, title in panels:
        corr = df[numeric_feats].corr()
        sns.heatmap(corr, ax=ax, cmap=cmap, vmin=-1, vmax=1,
                    annot=True, fmt='.2f', linewidths=0.4, linecolor='#EEEEEE',
                    xticklabels=feat_short, yticklabels=feat_short,
                    annot_kws={'size': 7}, square=True, cbar_kws={'shrink': 0.75})
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.tick_params(axis='x', rotation=0, labelsize=8)
        ax.tick_params(axis='y', rotation=0, labelsize=8)
    plt.tight_layout()
    save_fig('fig9_correlation_heatmaps.png')

def fig10_vocabulary(orig_tr, orig_te, proc_tr, proc_te):
    def vocab_stats(df, text_col):
        tc, uc, ttr_list, freq = [], [], [], Counter()
        for code in df[text_col].astype(str):
            tokens = code.lower().split()
            n = len(tokens); u = len(set(tokens))
            tc.append(n); uc.append(u)
            ttr_list.append(u / n if n > 0 else 0)
            freq.update(tokens)
        return np.array(tc), np.array(uc), np.array(ttr_list), freq
    ot_tc, ot_uc, ot_ttr, ot_fr = vocab_stats(orig_tr, ORIG_TEXT_COL)
    oe_tc, oe_uc, oe_ttr, oe_fr = vocab_stats(orig_te, ORIG_TEXT_COL)
    pt_tc, pt_uc, pt_ttr, pt_fr = vocab_stats(proc_tr, PROC_TEXT_COL)
    pe_tc, pe_uc, pe_ttr, pe_fr = vocab_stats(proc_te, PROC_TEXT_COL)
    fig = plt.figure(figsize=(22, 14), facecolor=BG)
    fig.suptitle('Figure 10  ·  Vocabulary & Lexical Richness Analysis',
                 fontsize=14, fontweight='bold', y=1.01, color=GREY_DARK)
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.4)
    ax0 = fig.add_subplot(gs[0, 0])
    for ttr, name, c in [(ot_ttr, LABEL_OT, COL_OT), (oe_ttr, LABEL_OE, COL_OE),
                          (pt_ttr, LABEL_PT, COL_PT), (pe_ttr, LABEL_PE, COL_PE)]:
        ax0.hist(ttr, bins=50, alpha=0.4, color=c, density=True, edgecolor='none')
        kde = stats.gaussian_kde(ttr)
        ax0.plot(np.linspace(0, 1, 300), kde(np.linspace(0, 1, 300)),
                 color=c, linewidth=2, label=name)
    ax0.set_xlabel('Type-Token Ratio (TTR)'); ax0.set_ylabel('Density')
    ax0.set_title('Lexical Diversity (TTR)'); ax0.legend(fontsize=7.5)
    ax1 = fig.add_subplot(gs[0, 1])
    for tc, uc, name, c in [(ot_tc, ot_uc, LABEL_OT, COL_OT), (oe_tc, oe_uc, LABEL_OE, COL_OE),
                              (pt_tc, pt_uc, LABEL_PT, COL_PT), (pe_tc, pe_uc, LABEL_PE, COL_PE)]:
        idx = np.random.choice(len(tc), min(1500, len(tc)), replace=False)
        ax1.scatter(tc[idx], uc[idx], alpha=0.2, color=c, s=7, label=name, edgecolors='none')
    ax1.set_xlabel('Total Tokens'); ax1.set_ylabel('Unique Tokens')
    ax1.set_title('Total vs. Unique Token Scatter'); ax1.legend(fontsize=7.5)
    ax1.xaxis.set_major_formatter(FuncFormatter(thousands_fmt))
    ax1.yaxis.set_major_formatter(FuncFormatter(thousands_fmt))
    ax2 = fig.add_subplot(gs[0, 2])
    for fr, name, c in [(ot_fr, LABEL_OT, COL_OT), (oe_fr, LABEL_OE, COL_OE),
                         (pt_fr, LABEL_PT, COL_PT), (pe_fr, LABEL_PE, COL_PE)]:
        rank = np.arange(1, min(500, len(fr)) + 1)
        freqs = sorted(fr.values(), reverse=True)[:500]
        ax2.loglog(rank, freqs, color=c, linewidth=2, label=name)
    ax2.set_xlabel('Token Rank (log)'); ax2.set_ylabel('Frequency (log)')
    ax2.set_title("Zipf's Law — Token Frequency"); ax2.legend(fontsize=7.5)
    ax2.grid(True, which='both', alpha=0.3)
    ax3 = fig.add_subplot(gs[1, 0])
    n_max = min(len(orig_tr), len(orig_te), len(proc_tr), len(proc_te))
    sizes = np.linspace(100, n_max, 20, dtype=int)
    for df, text_col, name, c in [
        (orig_tr, ORIG_TEXT_COL, LABEL_OT, COL_OT),
        (orig_te, ORIG_TEXT_COL, LABEL_OE, COL_OE),
        (proc_tr, PROC_TEXT_COL, LABEL_PT, COL_PT),
        (proc_te, PROC_TEXT_COL, LABEL_PE, COL_PE)]:
        growth = []
        for n in sizes:
            idx = np.random.choice(len(df), n, replace=False)
            v = len(set(' '.join(df[text_col].astype(str).iloc[idx]).lower().split()))
            growth.append(v)
        ax3.plot(sizes, growth, color=c, linewidth=2, label=name)
    ax3.set_xlabel('Sample Size'); ax3.set_ylabel('Vocabulary Size')
    ax3.set_title('Vocabulary Growth Curve'); ax3.legend(fontsize=7.5)
    ax3.xaxis.set_major_formatter(FuncFormatter(thousands_fmt))
    ax3.yaxis.set_major_formatter(FuncFormatter(thousands_fmt))
    ax4 = fig.add_subplot(gs[1, 1])
    top_n = 18
    all_terms = list((ot_fr + pt_fr).most_common(top_n))
    terms = [t for t, _ in all_terms]
    x = np.arange(len(terms)); w = 0.18
    for i, (fr, name, c) in enumerate([(ot_fr, LABEL_OT, COL_OT), (oe_fr, LABEL_OE, COL_OE),
                                         (pt_fr, LABEL_PT, COL_PT), (pe_fr, LABEL_PE, COL_PE)]):
        vals = [fr.get(t, 0) for t in terms]
        ax4.barh(x + (i - 1.5) * w, vals, w, color=c, label=name, edgecolor='white')
    ax4.set_yticks(x); ax4.set_yticklabels(terms, fontsize=8)
    ax4.set_xlabel('Frequency'); ax4.set_title(f'Top-{top_n} Tokens — All Splits')
    ax4.legend(fontsize=7.5)
    ax4.xaxis.set_major_formatter(FuncFormatter(thousands_fmt))
    ax5 = fig.add_subplot(gs[1, 2])
    rows = []
    for fr, tc, uc, ttr, name in [
        (ot_fr, ot_tc, ot_uc, ot_ttr, LABEL_OT),
        (oe_fr, oe_tc, oe_uc, oe_ttr, LABEL_OE),
        (pt_fr, pt_tc, pt_uc, pt_ttr, LABEL_PT),
        (pe_fr, pe_tc, pe_uc, pe_ttr, LABEL_PE)]:
        singletons = sum(1 for v in fr.values() if v == 1)
        rows.append([name, f'{len(fr):,}', f'{tc.mean():.0f}',
                     f'{np.median(tc):.0f}', f'{uc.mean():.0f}',
                     f'{ttr.mean():.4f}', f'{singletons:,}'])
    sdf = pd.DataFrame(rows, columns=['Split', 'Vocab', 'Avg Tok', 'Med Tok',
                                       'Avg Uniq', 'Avg TTR', 'Singletons'])
    draw_table(ax5, sdf, 'Vocabulary Statistics Summary')
    save_fig('fig10_vocabulary.png')

def fig11_drift_analysis(orig_tr_f, orig_te_f, proc_tr_f, proc_te_f,
                          tok_ot, tok_oe, tok_pt, tok_pe):
    features = ['char_len', 'line_count', 'word_count', 'lexical_density',
                'semicolons', 'operators', 'pointer_ops', 'func_calls_est', 'avg_line_len']
    flabels  = ['Char Len', 'Line Cnt', 'Word Cnt', 'Lex Density',
                'Semicolons', 'Operators', 'Ptr Ops', 'Func Calls', 'Avg Line']
    comparisons = [
        ('Orig: Train↔Test',   orig_tr_f, orig_te_f, tok_ot, tok_oe, COL_OT, COL_OE),
        ('Proc: Train↔Test',   proc_tr_f, proc_te_f, tok_pt, tok_pe, COL_PT, COL_PE),
        ('Train: Orig↔Proc',   orig_tr_f, proc_tr_f, tok_ot, tok_pt, COL_OT, COL_PT),
        ('Test:  Orig↔Proc',   orig_te_f, proc_te_f, tok_oe, tok_pe, COL_OE, COL_PE),
    ]
    fig = plt.figure(figsize=(22, 14), facecolor=BG)
    fig.suptitle('Figure 11  ·  Distribution Drift Analysis — Train↔Test & Original↔Processed',
                 fontsize=14, fontweight='bold', y=1.01, color=GREY_DARK)
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.5, wspace=0.42)
    all_rows = []
    for comp_name, df_a, df_b, tok_a, tok_b, ca, cb in comparisons:
        comp_rows = []
        for feat, fl in zip(features, flabels):
            va = df_a[feat].dropna().values
            vb = df_b[feat].dropna().values
            ks_s, ks_p = ks_2samp(va, vb)
            d = abs(cohen_d(va, vb))
            comp_rows.append({'Feature': fl, 'Comparison': comp_name,
                               'KS': ks_s, 'p': ks_p, 'd': d,
                               'neg_log_p': -np.log10(ks_p + 1e-300)})
        ks_s_tok, ks_p_tok = ks_2samp(tok_a, tok_b)
        comp_rows.append({'Feature': 'Token Len', 'Comparison': comp_name,
                           'KS': ks_s_tok, 'p': ks_p_tok,
                           'd': abs(cohen_d(tok_a, tok_b)),
                           'neg_log_p': -np.log10(ks_p_tok + 1e-300)})
        all_rows.extend(comp_rows)
    rdf = pd.DataFrame(all_rows)
    feat_order = flabels + ['Token Len']
    rdf['Feature'] = pd.Categorical(rdf['Feature'], categories=feat_order, ordered=True)
    rdf = rdf.sort_values('Feature')
    comp_colors = {c: col for c, col in zip([r[0] for r in comparisons],
                                             [COL_OE, COL_PE, COL_OT, ACCENT])}
    ax0 = fig.add_subplot(gs[0, 0])
    pivot_ks = rdf.pivot_table(index='Feature', columns='Comparison', values='KS')
    x = np.arange(len(pivot_ks)); w = 0.18
    for i, comp in enumerate([r[0] for r in comparisons]):
        if comp in pivot_ks.columns:
            ax0.bar(x + (i - 1.5) * w, pivot_ks[comp], w,
                    color=list(comp_colors.values())[i], label=comp,
                    edgecolor='white', alpha=0.85)
    ax0.axhline(0.05, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax0.set_xticks(x); ax0.set_xticklabels(pivot_ks.index, rotation=40, ha='right', fontsize=8)
    ax0.set_ylabel('KS Statistic'); ax0.set_title('KS Statistic per Feature × Comparison')
    ax0.legend(fontsize=7.5)
    ax1 = fig.add_subplot(gs[0, 1])
    pivot_nlp = rdf.pivot_table(index='Feature', columns='Comparison', values='neg_log_p')
    for i, comp in enumerate([r[0] for r in comparisons]):
        if comp in pivot_nlp.columns:
            ax1.bar(x + (i - 1.5) * w, pivot_nlp[comp], w,
                    color=list(comp_colors.values())[i], label=comp,
                    edgecolor='white', alpha=0.85)
    ax1.axhline(-np.log10(0.05), color=ACCENT, linestyle='--', linewidth=1.5, label='p=0.05')
    ax1.set_xticks(x); ax1.set_xticklabels(pivot_nlp.index, rotation=40, ha='right', fontsize=8)
    ax1.set_ylabel('-log₁₀(p)'); ax1.set_title('Significance per Feature × Comparison')
    ax1.legend(fontsize=7.5)
    ax2 = fig.add_subplot(gs[1, 0])
    pivot_d = rdf.pivot_table(index='Feature', columns='Comparison', values='d')
    for i, comp in enumerate([r[0] for r in comparisons]):
        if comp in pivot_d.columns:
            ax2.bar(x + (i - 1.5) * w, pivot_d[comp], w,
                    color=list(comp_colors.values())[i], label=comp,
                    edgecolor='white', alpha=0.85)
    for thresh, label in [(0.2, 'Small'), (0.5, 'Medium'), (0.8, 'Large')]:
        ax2.axhline(thresh, color='gray', linestyle=':', linewidth=0.9, alpha=0.7)
    ax2.set_xticks(x); ax2.set_xticklabels(pivot_d.index, rotation=40, ha='right', fontsize=8)
    ax2.set_ylabel("Cohen's |d|"); ax2.set_title("Effect Size per Feature × Comparison")
    ax2.legend(fontsize=7.5)
    ax3 = fig.add_subplot(gs[1, 1])
    summary_rows = []
    for comp_name, df_a, df_b, tok_a, tok_b, ca, cb in comparisons:
        sub = rdf[rdf['Comparison'] == comp_name]
        sig_feats = (sub['p'] < 0.05).sum()
        max_d     = sub['d'].max()
        mean_ks   = sub['KS'].mean()
        ks_tok, p_tok = ks_2samp(tok_a, tok_b)
        summary_rows.append([comp_name, f'{sig_feats}/{len(sub)}',
                              f'{mean_ks:.4f}', f'{max_d:.4f}',
                              f'{ks_tok:.4f}', f'{p_tok:.2e}'])
    sdf = pd.DataFrame(summary_rows, columns=['Comparison', 'Sig Feats',
                                               'Mean KS', 'Max |d|',
                                               'Token KS', 'Token p'])
    draw_table(ax3, sdf, 'Drift Summary Table — All Comparisons')
    save_fig('fig11_drift_analysis.png')

def fig12_outlier_summary(tok_ot, tok_oe, tok_pt, tok_pe,
                           orig_tr, orig_te, proc_tr, proc_te):
    tok_all = [tok_ot, tok_oe, tok_pt, tok_pe]
    fig = plt.figure(figsize=(22, 10), facecolor=BG)
    fig.suptitle('Figure 12  ·  Outlier & Extreme Sample Analysis',
                 fontsize=14, fontweight='bold', y=1.01, color=GREY_DARK)
    gs = gridspec.GridSpec(1, 3, figure=fig, hspace=0.4, wspace=0.4)
    ax0 = fig.add_subplot(gs[0, 0])
    outlier_pcts = []
    for tok, name, c in zip(tok_all, LABELS_4, PALETTE_4):
        q1, q3 = np.percentile(tok, [25, 75])
        upper = q3 + 1.5 * (q3 - q1)
        pct = np.mean(tok > upper) * 100
        outlier_pcts.append(pct)
    bars = ax0.bar(LABELS_4, outlier_pcts, color=PALETTE_4, edgecolor='white', width=0.55)
    for bar, val in zip(bars, outlier_pcts):
        ax0.text(bar.get_x() + bar.get_width() / 2, val + 0.1,
                 f'{val:.2f}%', ha='center', fontsize=10, fontweight='bold')
    ax0.set_ylabel('Outlier Samples (%)')
    ax0.set_title('IQR-Based Outlier Rate\n(token length > Q3 + 1.5·IQR)')
    ax0.set_xticklabels(LABELS_4, fontsize=8.5, rotation=12)
    ax1 = fig.add_subplot(gs[0, 1])
    hi_pcts = [90, 95, 97, 99, 99.5]
    for tok, name, c in zip(tok_all, LABELS_4, PALETTE_4):
        ax1.plot(hi_pcts, [np.percentile(tok, p) for p in hi_pcts],
                 marker='o', markersize=6, linewidth=2, color=c, label=name)
    ax1.set_xlabel('Percentile'); ax1.set_ylabel('Token Length')
    ax1.set_title('Extreme Percentile Profile'); ax1.legend(fontsize=8)
    ax1.yaxis.set_major_formatter(FuncFormatter(thousands_fmt))
    ax2 = fig.add_subplot(gs[0, 2])
    lo = min(t.min() for t in tok_all)
    hi = max(t.max() for t in tok_all)
    bins_log = np.logspace(np.log10(max(1, lo)), np.log10(hi), 60)
    for tok, name, c in zip(tok_all, LABELS_4, PALETTE_4):
        ax2.hist(tok, bins=bins_log, alpha=0.45, color=c, density=True,
                 edgecolor='none', label=name)
    ax2.set_xscale('log')
    ax2.set_xlabel('Token Length (log scale)'); ax2.set_ylabel('Density')
    ax2.set_title('Log-Scale Histogram\n(long-tail behaviour)')
    ax2.legend(fontsize=8)
    ax2.grid(True, which='both', alpha=0.3)
    save_fig('fig12_outlier_analysis.png')

def main():
    print('Loading datasets...')
    orig_tr, orig_te, proc_tr, proc_te = load_all()
    print('\nTokenizing — Original Train...')
    tok_ot = tokenize(orig_tr, ORIG_TEXT_COL)
    print('Tokenizing — Original Test...')
    tok_oe = tokenize(orig_te, ORIG_TEXT_COL)
    print('Tokenizing — Processed Train...')
    tok_pt = tokenize(proc_tr, PROC_TEXT_COL)
    print('Tokenizing — Processed Test...')
    tok_pe = tokenize(proc_te, PROC_TEXT_COL)
    datasets = [(LABEL_OT, orig_tr), (LABEL_OE, orig_te),
                (LABEL_PT, proc_tr), (LABEL_PE, proc_te)]
    print_summary(datasets, [tok_ot, tok_oe, tok_pt, tok_pe])
    print('\nComputing code features...')
    orig_tr_f = compute_features(orig_tr, ORIG_TEXT_COL)
    orig_te_f = compute_features(orig_te, ORIG_TEXT_COL)
    proc_tr_f = compute_features(proc_tr, PROC_TEXT_COL)
    proc_te_f = compute_features(proc_te, PROC_TEXT_COL)
    print('\nGenerating figures...')
    fig0_dataset_insights(orig_tr, orig_te, tok_ot, tok_oe)
    fig1_class_overview(orig_tr, orig_te, proc_tr, proc_te)
    fig2_token_overview(tok_ot, tok_oe, tok_pt, tok_pe)
    fig3_preprocessing_impact_tokens(tok_ot, tok_oe, tok_pt, tok_pe)
    fig4_class_stratified(orig_tr, orig_te, proc_tr, proc_te,
                           tok_ot, tok_oe, tok_pt, tok_pe)
    fig5_code_complexity(orig_tr_f, orig_te_f, proc_tr_f, proc_te_f)
    fig6_preprocessing_impact_features(orig_tr_f, orig_te_f, proc_tr_f, proc_te_f)
    fig7_class_complexity_contrast(orig_tr_f, proc_tr_f)
    fig8_statistical_significance(orig_tr_f, orig_te_f, proc_tr_f, proc_te_f)
    fig9_correlation_heatmaps(orig_tr_f, proc_tr_f)
    fig10_vocabulary(orig_tr, orig_te, proc_tr, proc_te)
    fig11_drift_analysis(orig_tr_f, orig_te_f, proc_tr_f, proc_te_f,
                          tok_ot, tok_oe, tok_pt, tok_pe)
    fig12_outlier_summary(tok_ot, tok_oe, tok_pt, tok_pe,
                           orig_tr, orig_te, proc_tr, proc_te)
    print('\n' + '=' * 80)
    print(f'ALL 13 FIGURES SAVED to folder: {OUT_DIR}/')
    print('=' * 80)

if __name__ == '__main__':
    main()